# 000_wfi_select_cals

This notebook is used to select datasets for calibration of WFI frames. 
The user should enter:

    - a directory where all the science data to be processed lies,
    - validity times for each of BIAS, DOME FLATS, and SKY_FLATS.
    - validity exposure times for sky flats

The script will output a list of all calibration frames needed by the mefpp 
data reduction engine.

Because WFGI does not have quality control Garching support it is not supported
by calSelector. Therefore, when data is retrieved from the archive no valid
calibration data can be attached. This script is design to help in the selection
of data. It cannot do QC on the data because it is not on disk yet. So, after
the script runs a notebook to do minimal quality controld on the calibration frames
should be run.

The QC note book will:

    - check level and dispersion of each bias and dome frames.
    - check exposure times of sky frames and only accept those with exptime < 60s (TBC)
    - compare sky frame levels with the prediction of models of
      sky brightness versus sun altitude. This will permit the determination of
      cloud presence.

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None) 
pd.set_option('display.max_rows', None)
from IPython.display import display
import numpy as np
from datetime import datetime, timedelta

In [ ]:
class ExecutionBlock_WFI:
    """
    Class to contain all exposures of a single template execution.
    It is constructed by passing as argument a list of exposures
    with the same TPL.START keyword. For BIAS exposures this is
    the only key that matters. For other DOME and SKY flat fields
    the exposures should have the same filter and (detector modes
    but these are not searchable in the ESO archive).
    """
    def __init__(self, exposure_list):
        self.exposure_list = exposure_list
        self.nexpo = len(exposure_list)
        self.tpl_start = exposure_list[0].tpl_start
        self.filter = exposure_list[0].filt

        self.exptime_min = 1000000.0
        self.exptime_max = -10000.0
        self.data_setid_list = []
        for expo in exposure_list:
            if expo.tpl_start != self.tpl_start:
                raise Exception("ExecutionBlock: discordant tpl_start")

            if expo.filt != self.filter:
                raise Exception("ExecutionBlock: discordant filter")

            if expo.exptime > self.exptime_max:
                self.exptime_max = expo.exptime

            if expo.exptime < self.exptime_min:
                self.exptime_min = expo.exptime 

            self.data_setid_list.append(expo.DatasetID)
        
class CalExp_WFI:
    """
    Class to contain information about a single exposure, from the archive.
    """
    def __init__(self, series):
        self.OBJECT = series['OBJECT']
        self.RA = series['RA']
        self.DEC = series['DEC']
        self.Program_ID = series['Program_ID']
        self.Instrument = series['Instrument']
        self.dpr_catg = series['Category']
        self.dpr_type = series['Type']
        self.mode = series['Mode']
        self.DatasetID = series['Dataset ID']
        self.release_date = series['Release_Date']
        self.tpl_id = series['TPL ID']
        self.tpl_start = series['TPL START']
        self.exptime = series['Exptime']
        self.filt_lbda_min = series['filter_lambda_min']
        self.filt_lbda_max = series['filter_lambda_max']
        self.filt = series['Filter']
        self.mjd_obs = series['MJD-OBS']
        self.airmass = series['Airmass']
        self.dimm_seeing = series['DIMM Seeing at Start']

In [ ]:
df = pd.read_csv('wfi_skyff_wdb_query_29964_eso.csv')

# here we remove rows which do not contain a value of TPL.START
df = df[df['TPL START'].str.strip().astype(bool)]
df.dropna(inplace=True)
for index, row in df.iterrows():
    try:
        pd.to_datetime(row['TPL START'])
    except:
        print(index, row)

df['Date'] = pd.to_datetime(df['TPL START']) # .dt.strftime('%Y-%m-%dT%H:%M:%S'))

#df_grouped = df.groupby(['Filter', 'TPL START'])
#print(df_grouped)

#df_skyff_Rc = df_grouped.get_group('BB#RC/162_ESO844')
#sel_skyff = (df['Category'] == 'CALIB') & (df['Type'] == 'FLAT,SKY')
#df_skyff = df[sel_skyff]
#df_skyff.info()
#row_count = df_skyff_Rc.shape[0]
#tpl_start_list = list(set(df_skyff_Rc['TPL START'].to_list()))
#tpl_start_list.sort()
#print("number of sky flat fields: {} - different tpl starts: {}".format(row_count, len(tpl_start_list)))
#exposure = CalExp_WFI(df.iloc[1])

#skyff_tpl_execs_RC_162 = {}
#for tpl_start in tpl_start_list:
#    select = (df['TPL START'] == tpl_start) & ('RC/162' in df['Filter'])
#    skyff_tpl_execs_RC_162[tpl_start] = df[select]

#print(len(skyff_tpl_execs_RC_162))

In [ ]:
# We select data according to time range of validity
skyff_date = datetime.strptime('2002-03-11T00:23:54', '%Y-%m-%dT%H:%M:%S')
start_date = skyff_date - timedelta(days=10)
end_date = skyff_date + timedelta(days=10)
filtered_df = df.loc[(start_date <= df['Date']) & (df['Date'] < end_date)]


# we partition by TPL.START and FILTER
n_by_tpl_start = filtered_df.groupby(["TPL START", "Filter"]) #.agg(["min", "max", "count"])

# we create a groups object
gb = n_by_tpl_start.groups


# Here we access the rows of the pandas dataframe group by group
execution_block_list = []
for key, value in gb.items():    
    if key[1] == "BB#V/89_ESO843":
        exposure_list = []
        df_partition = filtered_df.loc[gb[key]]
        for index, row in df_partition.iterrows():
            exposure_list.append(CalExp_WFI(row))
            
        #print(filtered_df.loc[gb[key]].shape[0])
        #display(filtered_df.loc[gb[key]])
        execution_block_list.append(ExecutionBlock_WFI(exposure_list))

for exec_block in execution_block_list:
    for adataset in exec_block.data_setid_list:
        print("{}    {} {} {} {} {}".format(adataset, exec_block.tpl_start, exec_block.nexpo, exec_block.filter, exec_block.exptime_min, exec_block.exptime_max))
    print("\n")